# VLM-Anomaly — PatchCore Full MVTec Sweep (Kaggle GPU)

**Model:** PatchCore via [Anomalib](https://github.com/openvinotoolkit/anomalib) — self-contained, no GitHub clone needed.  
**GPU:** T4 / P100 — set *Settings → Accelerator → GPU T4 x2*  
**Dataset:** Add [ipythonx/mvtec-ad](https://www.kaggle.com/datasets/ipythonx/mvtec-ad) via *Add Data*  
**Cost:** $0

## After the run
Download `patchcore_mvtec_results.json` from the **Output** tab, then locally:
```bash
cp ~/Downloads/patchcore_mvtec_results.json  <repo>/results/
# then run the report cell in any local notebook
```


In [ ]:
# ── Cell 1: Install anomalib (Kaggle ships torch/sklearn/pandas already) ────
# Pip will warn about conflicts with tensorflow/google-colab/etc. — those
# warnings are safe to ignore; none of those packages are used here.
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'anomalib==2.4.2', 'timm>=1.0.0', 'structlog>=24.4.0'],
    check=False,
)
import torch, anomalib
print(f'torch    : {torch.__version__}')
print(f'anomalib : {anomalib.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU      : {p.name}  ({p.total_memory/1e9:.1f} GB)')
else:
    print('WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU T4 x2')


In [ ]:
# ── Cell 2: Find MVTec dataset ───────────────────────────────────────────────
from pathlib import Path

MVTEC_ROOT = None
CANDIDATES = [
    Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'),
    Path('/kaggle/input/mvtec-ad'),
    Path('/kaggle/input/mvtec-anomaly-detection'),
    Path('/kaggle/input/mvtecad'),
]
EXPECTED = {'bottle', 'cable', 'capsule', 'carpet', 'grid'}

for c in CANDIDATES:
    if c.exists() and EXPECTED.issubset({d.name for d in c.iterdir() if d.is_dir()}):
        MVTEC_ROOT = c
        break

assert MVTEC_ROOT, (
    'MVTec not found. Add dataset ipythonx/mvtec-ad via Add Data.\n'
    f'Checked: {[str(c) for c in CANDIDATES]}'
)
categories = sorted(d.name for d in MVTEC_ROOT.iterdir() if d.is_dir())
print(f'MVTec root : {MVTEC_ROOT}')
print(f'Categories : {categories}')


In [ ]:
# ── Cell 3: Configure ────────────────────────────────────────────────────────
from pathlib import Path

MODEL_ID    = 'classical/patchcore'
IMAGE_SIZE  = 256
RESULTS_DIR = Path('/kaggle/working/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# anomalib's MVTecAD expects root/mvtec/{category}/ — symlink Kaggle input
_data = Path('/kaggle/working/data')
_data.mkdir(exist_ok=True)
_link = _data / 'mvtec'
if not _link.exists():
    _link.symlink_to(MVTEC_ROOT)
MVTEC_DATA_ROOT = str(_data)

def best_accelerator():
    """Use Lightning's own check — not torch.backends.mps (misleads on Intel Mac)."""
    try:
        from lightning.pytorch.accelerators import CUDAAccelerator, MPSAccelerator
        if CUDAAccelerator.is_available(): return 'gpu'
        if MPSAccelerator.is_available():  return 'mps'
    except Exception:
        pass
    return 'cpu'

ACCELERATOR = best_accelerator()
print(f'Accelerator : {ACCELERATOR}')
print(f'Results dir : {RESULTS_DIR}')


In [ ]:
# ── Cell 4: Inline evaluator ─────────────────────────────────────────────────
import json as _json, sys, time, uuid, warnings
warnings.filterwarnings('ignore')


def run_patchcore(category: str) -> dict:
    import torchvision.transforms.v2 as tv2
    from anomalib.data import MVTecAD
    from anomalib.engine import Engine
    from anomalib.models import Patchcore

    datamodule = MVTecAD(
        root=MVTEC_DATA_ROOT,
        category=category,
        train_batch_size=32,
        eval_batch_size=32,
        num_workers=4,
        augmentations=tv2.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    )
    model = Patchcore()
    engine = Engine(
        accelerator=ACCELERATOR,
        devices=1,
        max_epochs=1,
        enable_model_summary=False,
        enable_progress_bar=False,
    )

    _lim = sys.getrecursionlimit()
    sys.setrecursionlimit(5000)
    try:
        t0 = time.perf_counter()
        engine.fit(model, datamodule=datamodule)
        metrics_list = engine.test(model, datamodule=datamodule, verbose=False)
    finally:
        sys.setrecursionlimit(_lim)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    m = metrics_list[0] if metrics_list else {}
    def _get(key):
        for k in [key, f'test/{key}', f'image_{key}']:
            v = m.get(k)
            if v is not None:
                try: return float(v)
                except: pass
        return None

    result = {
        'model_id': MODEL_ID, 'backend': 'anomalib',
        'dataset': 'mvtec', 'category': category, 'n_images': 0,
        'auroc':     _get('image_AUROC') or _get('AUROC') or _get('auroc'),
        'f1':        _get('image_F1Score') or _get('F1Score') or _get('f1'),
        'precision': _get('image_Precision') or _get('Precision'),
        'recall':    _get('image_Recall') or _get('Recall'),
        'pro_score': None, 'mean_latency_ms': elapsed_ms, 'total_cost_usd': 0.0,
    }
    out = RESULTS_DIR / f'{uuid.uuid4().hex[:8]}_mvtec_{category}_patchcore.json'
    out.write_text(_json.dumps([result], indent=2))
    print(f'  {category:12s}  AUROC={result["auroc"] or 0:.3f}  '
          f'F1={result["f1"] or 0:.3f}  {elapsed_ms/1000:.0f}s')
    return result


def _already_done(cat):
    for f in RESULTS_DIR.glob(f'*_mvtec_{cat}_patchcore.json'):
        try:
            rows = _json.loads(f.read_text())
            if isinstance(rows, list) and any(r.get('model_id') == MODEL_ID for r in rows):
                return rows[0]
        except Exception:
            pass
    return None

print('Evaluator ready.')


In [ ]:
# ── Cell 5: Run all 15 categories (idempotent — skips already-done) ──────────
from tqdm.auto import tqdm

all_results = []
for category in tqdm(categories, desc=f'PatchCore [{ACCELERATOR}]'):
    done = _already_done(category)
    if done:
        print(f'  [skip] {category}')
        all_results.append(done)
    else:
        all_results.append(run_patchcore(category))

print(f'\nComplete: {len(all_results)}/15 categories.')


In [ ]:
# ── Cell 6: Save combined output & print download instructions ───────────────
import json as _json
from pathlib import Path

OUT = Path('/kaggle/working/patchcore_mvtec_results.json')
OUT.write_text(_json.dumps(all_results, indent=2))
print(f'Output : {OUT}  ({OUT.stat().st_size/1024:.1f} KB)')
print(f'Rows   : {len(all_results)}')
print()
print('1. Output tab (right panel) -> download patchcore_mvtec_results.json')
print('2. cp ~/Downloads/patchcore_mvtec_results.json <repo>/results/')
print('3. Run report cell locally to update REPORT.md')


In [ ]:
# ── Cell 7: Summary table ────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(all_results)
for col in ['auroc', 'f1', 'mean_latency_ms']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Mean AUROC  : {df.auroc.mean():.4f}')
print(f'Mean F1     : {df.f1.mean():.4f}')
print(f'Avg time/cat: {df.mean_latency_ms.mean()/1000:.0f}s')
print(f'Total cost  : $0.00')
print()
display(df[['category','auroc','f1','mean_latency_ms']].sort_values('auroc', ascending=False).reset_index(drop=True))
